In [0]:
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("activity_name", "")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
activity_name = dbutils.widgets.get("activity_name")

In [0]:
dbutils.widgets.text("run_date", "")
dbutils.widgets.text("days_back", "30")
dbutils.widgets.text("environment", "dev")

run_date = dbutils.widgets.get("run_date")
days_back = int(dbutils.widgets.get("days_back"))
environment = dbutils.widgets.get("environment")

print(f"Run date: {run_date}")
print(f"Days back: {days_back}")
print(f"Environment: {environment}")

Run date: 
Days back: 30
Environment: dev


In [0]:
# ------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------

from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

In [0]:
# ------------------------------------------
# 2. CONNECT TO ADLS USING KEY VAULT SECRET
# ------------------------------------------

storage_account_name = "staihospitaldev001"

storage_account_key = dbutils.secrets.get(
    scope="kv-aihoispital-scope",
    key="kv-aihospital-dev"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"

print("Gold path:", gold_path)

Gold path: abfss://gold@staihospitaldev001.dfs.core.windows.net/


In [0]:
from pyspark.sql import Row
from datetime import datetime

pipeline_run_id = dbutils.widgets.get("pipeline_run_id") if "pipeline_run_id" else ""
activity_name = dbutils.widgets.get("activity_name") if "activity_name" else ""

log_path = f"{gold_path}pipeline_logs"

def write_pipeline_log(stage_name, status, records_processed=0, message=""):
    log_df = spark.createDataFrame([
        Row(
            pipeline_run_id=pipeline_run_id,
            activity_name=activity_name,
            stage_name=stage_name,
            status=status,
            records_processed=int(records_processed),
            message=message,
            log_timestamp=datetime.utcnow()
        )
    ])
    
    log_df.write.format("delta").mode("append").save(log_path)

In [0]:
from pyspark.sql import Row
from datetime import datetime

# Widgets from ADF
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("activity_name", "")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
activity_name = dbutils.widgets.get("activity_name")

log_path = f"{gold_path}pipeline_logs"

def write_pipeline_log(stage_name, status, records_processed=0, message=""):
    log_df = spark.createDataFrame([
        Row(
            pipeline_run_id=pipeline_run_id,
            activity_name=activity_name,
            stage_name=stage_name,
            status=status,
            records_processed=int(records_processed),
            message=message,
            log_timestamp=datetime.utcnow()
        )
    ])
    
    log_df.write.format("delta").mode("append").save(log_path)

In [0]:
write_pipeline_log(
    stage_name="ML Training Scoring",
    status="Started",
    records_processed=0,
    message="ML training and scoring started"
)

In [0]:
# ------------------------------------------
# 3. READ GOLD ML FEATURE TABLE
# ------------------------------------------

ml_df = spark.read.format("delta").load(gold_path + "patient_incident_features")

print("Feature table row count:", ml_df.count())
display(ml_df)

Feature table row count: 1147


AdmissionID,PatientID,PatientAge,Gender,RiskCategory,WardID,WardName,Specialty,Capacity,AdmissionType,LengthOfStayDays,AdmissionStatus,LengthOfStayBand,IncidentCount,EscalatedIncidentCount,AvgSeverityScore,MaxSeverityScore,ShiftCoverageCount,DistinctStaffCount,TotalPlannedHours,NightShiftCount,StaffingPressureFlag,WillIncidentOccur,IsHighRiskPatient,IsEmergencyAdmission,IsActiveAdmission,AvgHoursPerStaff,IncidentRatePerDay,GenderCode,RiskCategoryCode
700000,10000,44,Male,Low,W02,Rehab & Recovery Ward,Rehabilitation,18,Planned,185,Active,90+ days,3,1,2.33,3,2705,220,25684,1011,0,1,0,0,1,116.75,0.0162,1,1
700001,10001,36,Female,Low,W04,Older Adults Ward,Geriatric MH,20,Transfer,18,Discharged,8-30 days,0,0,0.0,0,3267,220,31228,1273,0,0,0,0,0,141.95,0.0,0,1
700002,10001,36,Female,Low,W03,Secure Unit,Forensic,16,Transfer,160,Active,90+ days,0,0,0.0,0,2990,220,28576,1164,0,0,0,0,1,129.89,0.0,0,1
700003,10002,47,Male,Medium,W02,Rehab & Recovery Ward,Rehabilitation,18,Emergency,315,Active,90+ days,0,0,0.0,0,2705,220,25684,1011,0,0,0,1,1,116.75,0.0,1,2
700004,10002,47,Male,Medium,W01,Acute Psychiatric Ward,Mental Health,24,Emergency,13,Discharged,8-30 days,1,0,1.0,1,4947,220,46940,1841,0,1,0,1,0,213.36,0.0769,1,2
700005,10003,59,Male,Medium,W02,Rehab & Recovery Ward,Rehabilitation,18,Planned,20,Discharged,8-30 days,0,0,0.0,0,2705,220,25684,1011,0,0,0,0,0,116.75,0.0,1,2
700006,10003,59,Male,Medium,W04,Older Adults Ward,Geriatric MH,20,Emergency,13,Discharged,8-30 days,1,1,2.0,2,3267,220,31228,1273,0,1,0,1,0,141.95,0.0769,1,2
700007,10003,59,Male,Medium,W02,Rehab & Recovery Ward,Rehabilitation,18,Emergency,288,Active,90+ days,0,0,0.0,0,2705,220,25684,1011,0,0,0,1,1,116.75,0.0,1,2
700008,10004,34,Male,Low,W04,Older Adults Ward,Geriatric MH,20,Emergency,294,Active,90+ days,0,0,0.0,0,3267,220,31228,1273,0,0,0,1,1,141.95,0.0,1,1
700009,10005,34,Male,Low,W04,Older Adults Ward,Geriatric MH,20,Planned,60,Active,31-90 days,0,0,0.0,0,3267,220,31228,1273,0,0,0,0,1,141.95,0.0,1,1


In [0]:
# ------------------------------------------
# 4. SELECT TRAINING COLUMNS AND REMOVE LEAKAGE
# ------------------------------------------

training_df = ml_df.select(
    "AdmissionID",
    "PatientID",
    "PatientAge",
    "Gender",
    "RiskCategory",
    "WardName",
    "Specialty",
    "Capacity",
    "AdmissionType",
    "LengthOfStayDays",
    "AdmissionStatus",
    "LengthOfStayBand",
    "ShiftCoverageCount",
    "DistinctStaffCount",
    "TotalPlannedHours",
    "NightShiftCount",
    "StaffingPressureFlag",
    "WillIncidentOccur"
)

display(training_df)

AdmissionID,PatientID,PatientAge,Gender,RiskCategory,WardName,Specialty,Capacity,AdmissionType,LengthOfStayDays,AdmissionStatus,LengthOfStayBand,ShiftCoverageCount,DistinctStaffCount,TotalPlannedHours,NightShiftCount,StaffingPressureFlag,WillIncidentOccur
700000,10000,44,Male,Low,Rehab & Recovery Ward,Rehabilitation,18,Planned,185,Active,90+ days,2705,220,25684,1011,0,1
700001,10001,36,Female,Low,Older Adults Ward,Geriatric MH,20,Transfer,18,Discharged,8-30 days,3267,220,31228,1273,0,0
700002,10001,36,Female,Low,Secure Unit,Forensic,16,Transfer,160,Active,90+ days,2990,220,28576,1164,0,0
700003,10002,47,Male,Medium,Rehab & Recovery Ward,Rehabilitation,18,Emergency,315,Active,90+ days,2705,220,25684,1011,0,0
700004,10002,47,Male,Medium,Acute Psychiatric Ward,Mental Health,24,Emergency,13,Discharged,8-30 days,4947,220,46940,1841,0,1
700005,10003,59,Male,Medium,Rehab & Recovery Ward,Rehabilitation,18,Planned,20,Discharged,8-30 days,2705,220,25684,1011,0,0
700006,10003,59,Male,Medium,Older Adults Ward,Geriatric MH,20,Emergency,13,Discharged,8-30 days,3267,220,31228,1273,0,1
700007,10003,59,Male,Medium,Rehab & Recovery Ward,Rehabilitation,18,Emergency,288,Active,90+ days,2705,220,25684,1011,0,0
700008,10004,34,Male,Low,Older Adults Ward,Geriatric MH,20,Emergency,294,Active,90+ days,3267,220,31228,1273,0,0
700009,10005,34,Male,Low,Older Adults Ward,Geriatric MH,20,Planned,60,Active,31-90 days,3267,220,31228,1273,0,0


In [0]:
# ------------------------------------------
# 5. HANDLE NULL VALUES
# ------------------------------------------

training_df = training_df.fillna({
    "PatientAge": 0,
    "Gender": "Unknown",
    "RiskCategory": "Unknown",
    "WardName": "Unknown",
    "Specialty": "Unknown",
    "Capacity": 0,
    "AdmissionType": "Unknown",
    "AdmissionStatus": "Unknown",
    "LengthOfStayBand": "Unknown",
    "LengthOfStayDays": 0,
    "ShiftCoverageCount": 0,
    "DistinctStaffCount": 0,
    "TotalPlannedHours": 0,
    "NightShiftCount": 0,
    "StaffingPressureFlag": 0
})

In [0]:
# ------------------------------------------
# 6. ENCODE CATEGORICAL FEATURES
# ------------------------------------------

gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex", handleInvalid="keep")
risk_indexer = StringIndexer(inputCol="RiskCategory", outputCol="RiskCategoryIndex", handleInvalid="keep")
ward_indexer = StringIndexer(inputCol="WardName", outputCol="WardNameIndex", handleInvalid="keep")
specialty_indexer = StringIndexer(inputCol="Specialty", outputCol="SpecialtyIndex", handleInvalid="keep")
admission_type_indexer = StringIndexer(inputCol="AdmissionType", outputCol="AdmissionTypeIndex", handleInvalid="keep")
admission_status_indexer = StringIndexer(inputCol="AdmissionStatus", outputCol="AdmissionStatusIndex", handleInvalid="keep")
los_band_indexer = StringIndexer(inputCol="LengthOfStayBand", outputCol="LengthOfStayBandIndex", handleInvalid="keep")

In [0]:
# ------------------------------------------
# 7. ASSEMBLE MODEL FEATURES
# ------------------------------------------

feature_columns = [
    "PatientAge",
    "GenderIndex",
    "RiskCategoryIndex",
    "WardNameIndex",
    "SpecialtyIndex",
    "Capacity",
    "AdmissionTypeIndex",
    "AdmissionStatusIndex",
    "LengthOfStayBandIndex",
    "LengthOfStayDays",
    "ShiftCoverageCount",
    "DistinctStaffCount",
    "TotalPlannedHours",
    "NightShiftCount",
    "StaffingPressureFlag"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

In [0]:
# ------------------------------------------
# 8. DEFINE THE ML MODEL
# ------------------------------------------

lr = LogisticRegression(
    featuresCol="features",
    labelCol="WillIncidentOccur",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=30
)

In [0]:
# ------------------------------------------
# 9. BUILD ML PIPELINE
# ------------------------------------------

pipeline = Pipeline(stages=[
    gender_indexer,
    risk_indexer,
    ward_indexer,
    specialty_indexer,
    admission_type_indexer,
    admission_status_indexer,
    los_band_indexer,
    assembler,
    lr
])

In [0]:
# ------------------------------------------
# 10. SPLIT TRAIN AND TEST DATA
# ------------------------------------------

train_df, test_df = training_df.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

Train rows: 948
Test rows: 199


In [0]:
# ------------------------------------------
# 11. TRAIN THE MODEL
# ------------------------------------------

model = pipeline.fit(train_df)

print("Model training complete.")

Model training complete.


In [0]:
# ------------------------------------------
# 12. GENERATE PREDICTIONS
# ------------------------------------------

predictions = model.transform(test_df)

display(
    predictions.select(
        "AdmissionID",
        "PatientID",
        "WillIncidentOccur",
        "prediction",
        "probability"
    )
)

AdmissionID,PatientID,WillIncidentOccur,prediction,probability
700002,10001,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6892570453795098, 0.3107429546204902))"
700006,10003,1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.71174835673175, 0.28825164326825004))"
700008,10004,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.5666259791427393, 0.43337402085726073))"
700013,10007,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.8168878975141793, 0.18311210248582066))"
700019,10011,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.5920341653649432, 0.40796583463505676))"
700023,10014,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6542207173753191, 0.34577928262468094))"
700029,10020,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.7391692796919073, 0.26083072030809273))"
700035,10023,1,1.0,"Map(vectorType -> dense, length -> 2, values -> List(0.3909315856655671, 0.6090684143344329))"
700045,10033,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6820575002868657, 0.31794249971313426))"
700046,10033,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6817238244339722, 0.31827617556602783))"


In [0]:
# ------------------------------------------
# 13. EVALUATE MODEL PERFORMANCE
# ------------------------------------------

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="WillIncidentOccur",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="WillIncidentOccur",
    predictionCol="prediction",
    metricName="accuracy"
)

auc = auc_evaluator.evaluate(predictions)
accuracy = accuracy_evaluator.evaluate(predictions)

print("AUC:", round(auc, 4))
print("Accuracy:", round(accuracy, 4))

AUC: 0.6616
Accuracy: 0.6533


In [0]:
predictions.groupBy("WillIncidentOccur", "prediction").count().show()

+-----------------+----------+-----+
|WillIncidentOccur|prediction|count|
+-----------------+----------+-----+
|                1|       0.0|   51|
|                0|       0.0|  107|
|                1|       1.0|   23|
|                0|       1.0|   18|
+-----------------+----------+-----+



In [0]:
# ------------------------------------------
# 14. SCORE FULL DATASET FOR REPORTING
# ------------------------------------------

full_predictions = model.transform(training_df)

display(
    full_predictions.select(
        "AdmissionID",
        "PatientID",
        "WillIncidentOccur",
        "prediction",
        "probability"
    )
)

# COMMAND ----------

# ------------------------------------------
# 15. BUILD REPORTING PREDICTIONS TABLE
# ------------------------------------------

gold_predictions = (
    full_predictions
    .select(
        "AdmissionID",
        "PatientID",
        "WillIncidentOccur",
        F.col("prediction").cast("int").alias("PredictedIncidentFlag_Default"),
        vector_to_array("probability")[1].alias("PredictedIncidentProbability")
    )
    .withColumn(
        "PredictedIncidentFlag_30",
        F.when(F.col("PredictedIncidentProbability") >= 0.30, 1).otherwise(0)
    )
    .withColumn(
        "PredictedIncidentFlag_40",
        F.when(F.col("PredictedIncidentProbability") >= 0.40, 1).otherwise(0)
    )
    .withColumn(
        "PredictedIncidentFlag_50",
        F.when(F.col("PredictedIncidentProbability") >= 0.50, 1).otherwise(0)
    )
)

display(gold_predictions)

# COMMAND ----------

# ------------------------------------------
# 16. WRITE PREDICTIONS TO GOLD
# ------------------------------------------

gold_predictions.write.format("delta").mode("overwrite").save(
    gold_path + "patient_incident_predictions"
)

print("Full-dataset predictions written successfully.")

# COMMAND ----------

# ------------------------------------------
# 17. VALIDATE SAVED PREDICTIONS
# ------------------------------------------

saved_predictions = spark.read.format("delta").load(
    gold_path + "patient_incident_predictions"
)

print("Saved predictions row count:", saved_predictions.count())
display(saved_predictions)

AdmissionID,PatientID,WillIncidentOccur,prediction,probability
700000,10000,1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.719421343627769, 0.280578656372231))"
700001,10001,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.8534053881910639, 0.14659461180893607))"
700002,10001,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6892570453795098, 0.3107429546204902))"
700003,10002,0,1.0,"Map(vectorType -> dense, length -> 2, values -> List(0.43301596890785987, 0.5669840310921401))"
700004,10002,1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6390805230433053, 0.3609194769566947))"
700005,10003,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.7642318339110631, 0.23576816608893691))"
700006,10003,1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.71174835673175, 0.28825164326825004))"
700007,10003,0,1.0,"Map(vectorType -> dense, length -> 2, values -> List(0.4659862732154356, 0.5340137267845644))"
700008,10004,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.5666259791427393, 0.43337402085726073))"
700009,10005,0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.791523016893688, 0.20847698310631202))"


AdmissionID,PatientID,WillIncidentOccur,PredictedIncidentFlag_Default,PredictedIncidentProbability,PredictedIncidentFlag_30,PredictedIncidentFlag_40,PredictedIncidentFlag_50
700000,10000,1,0,0.280578656372231,0,0,0
700001,10001,0,0,0.14659461180893607,0,0,0
700002,10001,0,0,0.3107429546204902,1,0,0
700003,10002,0,1,0.5669840310921401,1,1,1
700004,10002,1,0,0.3609194769566947,1,0,0
700005,10003,0,0,0.23576816608893691,0,0,0
700006,10003,1,0,0.28825164326825004,0,0,0
700007,10003,0,1,0.5340137267845644,1,1,1
700008,10004,0,0,0.43337402085726073,1,1,0
700009,10005,0,0,0.20847698310631202,0,0,0


Full-dataset predictions written successfully.
Saved predictions row count: 1147


AdmissionID,PatientID,WillIncidentOccur,PredictedIncidentFlag_Default,PredictedIncidentProbability,PredictedIncidentFlag_30,PredictedIncidentFlag_40,PredictedIncidentFlag_50
700000,10000,1,0,0.280578656372231,0,0,0
700001,10001,0,0,0.14659461180893607,0,0,0
700002,10001,0,0,0.3107429546204902,1,0,0
700003,10002,0,1,0.5669840310921401,1,1,1
700004,10002,1,0,0.3609194769566947,1,0,0
700005,10003,0,0,0.23576816608893691,0,0,0
700006,10003,1,0,0.28825164326825004,0,0,0
700007,10003,0,1,0.5340137267845644,1,1,1
700008,10004,0,0,0.43337402085726073,1,1,0
700009,10005,0,0,0.20847698310631202,0,0,0


In [0]:
# ------------------------------------------
# AZURE SQL CONNECTION
# ------------------------------------------

sql_scope = "kv-aihospital-sql-scope"

# Hardcoded values
sql_server = "aihospital-sql-dev.database.windows.net"
sql_database = "aihospital-sql-dev"   # use the actual database name shown in Azure
sql_username = "aihospital-sql-dev"

# Password from Key Vault via Databricks secret scope
sql_password = dbutils.secrets.get(
    scope=sql_scope,
    key="azure-sql-password"
)

jdbc_url = (
    f"jdbc:sqlserver://{sql_server}:1433;"
    f"database={sql_database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user": sql_username,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print("Azure SQL connection config loaded.")
print("Server:", sql_server)
print("Database:", sql_database)
print("Username:", sql_username)

Azure SQL connection config loaded.
Server: aihospital-sql-dev.database.windows.net
Database: aihospital-sql-dev
Username: aihospital-sql-dev


In [0]:
# ------------------------------------------
# FUNCTION TO WRITE PREDICTIONS TABLE TO AZURE SQL
# ------------------------------------------
def write_to_azure_sql(df, table_name, mode="overwrite"):
    df.write.jdbc(
        url=jdbc_url,
        table=table_name,
        mode=mode,
        properties=connection_properties
    )
    print(f"Write to {table_name} completed successfully.")

In [0]:
# ------------------------------------------
# WRITE PREDICTIONS TABLE TO AZURE SQL
# ------------------------------------------

write_to_azure_sql(
    gold_predictions,
    "dbo.patient_incident_predictions",
    mode="overwrite"
)

Write to dbo.patient_incident_predictions completed successfully.


In [0]:
ml_record_count = gold_predictions.count()

write_pipeline_log(
    stage_name="ML Training Scoring",
    status="Succeeded",
    records_processed=ml_record_count,
    message="ML model training, scoring and SQL publishing completed successfully"
)